# Online Learning — No-Clip Solution

Target clipping removed entirely (`y_clip=None`). The protection stack that replaces it:

| mechanism | job |
|---|---|
| `scale_floor` (train-derived, q0.15 of window stds ≈ 0.095) | flat lookback windows can't explode `y_scaled` (max drops from 12.2M to ~122) |
| `max_grad_norm=1.0` (default in `train_model_online`) | one sample can never move the weights more than a bounded step; runs **before** the optimizer, so Adam/FTRL state never sees a spike |
| raw-`y` ground truth (built into the 5-tuple datasets) | metrics scored against the real gauge, never a censored reconstruction |

Measured beforehand (`diagnostics/`, scratch experiment 2026-07-17): with the floor, a clamp at 500 binds on **0 of 76,536** targets, so removing it entirely is tensor-identical — this notebook should reproduce the fixed-pipeline numbers exactly:

| model | expected RMSE / R² (seed 42) |
|---|---|
| FTRL-default | 4.836 / 0.823 |
| RMSprop | 5.702 / 0.754 |
| Adam-default | 6.224 / 0.707 |

**Why keep the floor even without a clip:** no-floor + no-clip survives only because gradient clipping sits upstream — but it gives up float32 overflow safety (a corrupted reading could produce an `inf` loss → `NaN` weights mid-stream) and trains flat-window samples on meaningless targets. The floor costs nothing and closes both holes.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

import util_fun as uf
from cstm_models import build_mlp
from cstm_models.ftrl import FTRL

In [2]:
# model parameters
LOOKBACK = 48   # 48 h of historical data for prediction
HORIZON = 24    # 24 h of prediction
N_FEATURES = 4  # Q_bar, P_bar, T_bar, H_bar
BATCH_SIZE = 1
H_BAR_INDEX = 3

In [3]:
# timestamps are ISO — parse WITHOUT dayfirst (dayfirst=True silently leaves the index as strings)
train_df = pd.read_csv('dataset/one_station_train_data.csv', parse_dates=[0], index_col=0)
test_df = pd.read_csv('dataset/one_station_test_data.csv', parse_dates=[0], index_col=0)

x_train, y_train = uf.create_sequences(train_df.values.astype(np.float32), LOOKBACK, HORIZON, H_BAR_INDEX)
x_test, y_test = uf.create_sequences(test_df.values.astype(np.float32), LOOKBACK, HORIZON, H_BAR_INDEX)

Creating sequences...
Data length: 59123, lookback: 48, horizon: 24 , step: 24
End of sufficient data at index 59064.
Total sequences created: 2461
 X shape: (2461, 48, 4), y shape: (2461, 24)
Creating sequences...
Data length: 17524, lookback: 48, horizon: 24 , step: 24
End of sufficient data at index 17472.
Total sequences created: 728
 X shape: (728, 48, 4), y shape: (728, 24)


## No-clip datasets

`y_clip=None` — no clamp anywhere in the target path. The scale floor is the only guard, computed from TRAIN windows only (a fixed constant → no leakage; do not tune the quantile against test metrics).

In [4]:
scale_floor = uf.compute_scale_floor(x_train, H_BAR_INDEX, quantile=0.15)

train_ds_noclip = uf.RollingNormTimeSeriesDataset(x_train, y_train, H_BAR_INDEX,
                                                  y_clip=None, scale_floor=scale_floor)
test_ds_noclip  = uf.RollingNormTimeSeriesDataset(x_test,  y_test,  H_BAR_INDEX,
                                                  y_clip=None, scale_floor=scale_floor)

uf.report_y_clip_binding(train_ds_noclip, 'train')
uf.report_y_clip_binding(test_ds_noclip, 'test')

scale_floor = 0.094705 (q0.15 of 2461 train-window stds; median 0.5635)
y_clip binding [train]: no clip configured (y_clip=None) — nothing can bind
y_clip binding [test]: no clip configured (y_clip=None) — nothing can bind


0.0

## Train + evaluate the three headline models

Same seeds and configs as the fixed pipeline (`rerun_fixed_pipeline.py`); `max_grad_norm=1.0` is the default now, so every run is gradient-clipped.

In [5]:
os.makedirs('results/testing/online_noclip', exist_ok=True)

noclip_results = {}
for label, make_opt in [
    ('Adam-default', lambda m: optim.Adam(m.parameters(), lr=1e-3)),
    ('RMSprop',      lambda m: optim.RMSprop(m.parameters(), lr=1e-3, alpha=0.99)),
    ('FTRL-default', lambda m: FTRL(m.parameters(), alpha=0.01, beta=1.0, lambda1=0.0, lambda2=0.0)),
]:
    print(f'Running {label}...', end=' ', flush=True)
    torch.manual_seed(42)
    model_nc, _, crit_nc = build_mlp(LOOKBACK * N_FEATURES, HORIZON)
    opt_nc = make_opt(model_nc)

    uf.train_model_online(model_nc, opt_nc, crit_nc, train_ds_noclip, train_df,
                          LOOKBACK, HORIZON, BATCH_SIZE, silent=True)
    y_te_nc, y_pe_nc, idx_te_nc = uf.model_evaluate_with_norm(
        model_nc, crit_nc, test_ds_noclip, test_df.index, LOOKBACK, HORIZON, BATCH_SIZE
    )
    noclip_results[label] = (y_te_nc, y_pe_nc, idx_te_nc)
    uf.export_results_to_csv({label: (y_te_nc, y_pe_nc, idx_te_nc)},
                             'results/testing/online_noclip/online_noclip')

Running Adam-default... Testing complete.
Exported results to CSV: results/testing/online_noclip/online_noclip_Adam-default.csv
Running RMSprop... Testing complete.
Exported results to CSV: results/testing/online_noclip/online_noclip_RMSprop.csv
Running FTRL-default... Testing complete.
Exported results to CSV: results/testing/online_noclip/online_noclip_FTRL-default.csv


In [ ]:
uf.calculate_metrics_and_plot(
    noclip_results,
    plot_title='No-Clip Solution (scale floor, no clamp) — Test Set',
    export_metrics=True, export_html=True,
    export_file_name='results/testing/online/noclip_comparison',
)

## Sanity check — no-clip must match the fixed pipeline exactly

The tripwire (`y_clip=500`) binds 0%, so the clamp was a no-op: predictions from this notebook and from `rerun_fixed_pipeline.py` (`results/testing/online_fixed/`) should agree to float precision. If this check ever fails, something upstream changed.

In [ ]:
for label in noclip_results:
    fixed_path = f'results/testing/online_fixed/online_fixed_{label}.csv'
    if not os.path.exists(fixed_path):
        print(f'{label}: fixed-pipeline CSV not found, skipped')
        continue
    fixed = pd.read_csv(fixed_path)
    _, y_pe_nc, _ = noclip_results[label]
    max_diff = np.abs(np.sort(fixed['Prediction'].values) - np.sort(np.asarray(y_pe_nc, dtype=float))).max()
    print(f'{label}: max |no-clip - fixed| prediction diff = {max_diff:.2e}')